In [1]:
import sys
from ortools.sat.python import cp_model

In [5]:
def read_input():
    data = list(map(int, sys.stdin.read().split()))
    n, m = data[0], data[1]
    idx = 2
    edges = []
    
    for _ in range(m):
        u, v, w = data[idx], data[idx + 1], data[idx + 2]
        idx += 3
        edges.append((u,v,w))

    return n, edges

In [6]:
def solve(n, edges):
    model = cp_model.CpModel()
    x = [model.new_bool_var(f"x[{i}]") for i in range(n)]
    model.add(sum(x[i] for i in range(n)) == int(n / 2))
    total_weights = []
    for u, v, w in edges:
        check = model.new_bool_var(f"check_{u}_{v}")
        model.add(x[u] != x[v]).only_enforce_if(check)
        model.add(x[u] == x[v]).only_enforce_if(check.Not())
        total_weights.append(check * w)

    model.minimize(sum(total_weights))
    solver = cp_model.CpSolver()
    
    solver.parameters.max_time_in_seconds = 10.0

    status = solver.solve(model)
    if status in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print(solver.objective_value)

In [7]:
def main():
    f = open("input.txt", "r")
    sys.stdin = f
    n, edges = read_input()
    solve(n, edges)
    f.close()

main()

5.0
